# Alchemy GeomML + TDA — Results Visualization

Графики обучения, сравнение моделей, анализ TDA.

**Данные:** v29 (bs=512, полный датасет, 4 модели) + v33.10 (bs=1024, метрики)

In [ ]:
import sys, os, glob, json
sys.path.insert(0, 'src')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

try:
    fm.fontManager.addfont('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf')
except: pass
plt.rcParams['font.sans-serif'] = ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

os.makedirs('results/figures/v34', exist_ok=True)

# Загрузка v29 history
v29_dir = 'results/experiments/batch_size_512'
histories = {}
for csv in sorted(glob.glob(f'{v29_dir}/history_*.csv')):
    name = csv.split('/')[-1].replace('history_', '').rsplit('_', 2)[0]
    histories[name] = pd.read_csv(csv)
    print(f'Loaded: {name} ({len(histories[name])} epochs)')

# v29 summary
summary_v29 = pd.read_csv(f'{v29_dir}/summary_all_v29_4.csv')
print(f'\nv29 summary:\n{summary_v29.to_string(index=False)}')

## 1. Training Curves (v29, bs=512)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12), constrained_layout=True)
fig.suptitle('Training Curves — v29 (bs=512, full Alchemy dataset)', fontsize=16, fontweight='bold')

colors = {'egnn': '#2196F3', 'egnn_tda': '#FF9800', 'egnn_vector': '#4CAF50', 'egnn_vector_tda': '#9C27B0'}
labels = {'egnn': 'EGNN', 'egnn_tda': 'EGNN+TDA', 'egnn_vector': 'EGNN Vector', 'egnn_vector_tda': 'EGNN Vector+TDA'}

# Loss
ax = axes[0, 0]
for name, df in histories.items():
    ax.plot(df['epoch'], df['train_loss'], '--', color=colors[name], alpha=0.5, linewidth=1)
    ax.plot(df['epoch'], df['val_loss'], '-', color=colors[name], label=labels[name], linewidth=2)
ax.set_title('Loss (normalized MAE, sum of 3 targets)', fontsize=13)
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.legend(); ax.grid(True, alpha=0.3)
ax.set_yscale('log')

# mu MAE
ax = axes[0, 1]
for name, df in histories.items():
    if 'val_mu_mae' in df.columns:
        ax.plot(df['epoch'], df['val_mu_mae'], '-', color=colors[name], label=labels[name], linewidth=2)
ax.set_title('μ MAE (Debye)', fontsize=13)
ax.set_xlabel('Epoch'); ax.set_ylabel('MAE')
ax.legend(); ax.grid(True, alpha=0.3)

# alpha MAE
ax = axes[1, 0]
for name, df in histories.items():
    if 'val_alpha_mae' in df.columns:
        ax.plot(df['epoch'], df['val_alpha_mae'], '-', color=colors[name], label=labels[name], linewidth=2)
ax.set_title('α MAE (Bohr³)', fontsize=13)
ax.set_xlabel('Epoch'); ax.set_ylabel('MAE')
ax.legend(); ax.grid(True, alpha=0.3)

# gap MAE
ax = axes[1, 1]
for name, df in histories.items():
    if 'val_gap_mae' in df.columns:
        ax.plot(df['epoch'], df['val_gap_mae'], '-', color=colors[name], label=labels[name], linewidth=2)
ax.set_title('gap MAE (Hartree)', fontsize=13)
ax.set_xlabel('Epoch'); ax.set_ylabel('MAE')
ax.legend(); ax.grid(True, alpha=0.3)

plt.savefig('results/figures/v34/training_curves_v29.png', dpi=150, facecolor='white')
plt.show()
print('Saved: results/figures/v34/training_curves_v29.png')

## 2. Test Metrics Comparison (Bar Chart)

In [ ]:
# Все модели: v26.1 (bs=256) + v29 (bs=512) + v33.10 (bs=1024)
all_results = []

# v26.1 baselines
for model, csv_path in [('FCNN', 'results/experiments/batch_size_256/history_fcnn_all.csv'),
                         ('SchNet', 'results/experiments/batch_size_256/history_schnet_all.csv'),
                         ('EGNN', 'results/experiments/batch_size_256/history_egnn_all.csv')]:
    df = pd.read_csv(csv_path)
    last = df.iloc[-1]
    all_results.append({'model': model, 'version': 'v26.1 (bs=256)',
                        'mu_mae': last['test_mu_mae'], 'alpha_mae': last['test_alpha_mae'],
                        'gap_mae': last['test_gap_mae'], 'test_loss': last['test_loss'],
                        'epochs': len(df), 'params': {'FCNN': 53171, 'SchNet': 500000, 'EGNN': 951759}[model]})

# v29 EGNN family
for _, row in summary_v29.iterrows():
    name = row['model'].replace('_', '+')
    params = {'egnn': 951759, 'egnn+tda': 952811, 'egnn+vector': 968023, 'egnn+vector+tda': 981439}.get(name, 950000)
    all_results.append({'model': name, 'version': 'v29 (bs=512)',
                        'mu_mae': row['test_mu_mae'], 'alpha_mae': row['test_alpha_mae'],
                        'gap_mae': row['test_gap_mae'], 'test_loss': row['test_loss'],
                        'epochs': row['n_epochs'], 'params': params})

# v33.10 EGNN (bs=1024)
all_results.append({'model': 'EGNN', 'version': 'v33.10 (bs=1024)',
                    'mu_mae': 0.286, 'alpha_mae': 0.574, 'gap_mae': 0.0058,
                    'test_loss': 0.344, 'epochs': 110, 'params': 951759})

df_all = pd.DataFrame(all_results)
print(df_all.to_string(index=False))

# Bar chart
fig, axes = plt.subplots(1, 3, figsize=(18, 6), constrained_layout=True)
fig.suptitle('Test MAE Comparison Across All Models', fontsize=16, fontweight='bold')

for ax, (metric, title, unit) in zip(axes, [
    ('mu_mae', 'μ MAE', 'Debye'),
    ('alpha_mae', 'α MAE', 'Bohr³'),
    ('gap_mae', 'gap MAE', 'Hartree')
]):
    # Только v29 models для сравнения
    v29_data = df_all[df_all['version'] == 'v29 (bs=512)']
    bars = ax.bar(range(len(v29_data)), v29_data[metric], color=['#2196F3', '#FF9800', '#4CAF50', '#9C27B0'])
    ax.set_xticks(range(len(v29_data)))
    ax.set_xticklabels(['EGNN', 'EGNN+TDA', 'EGNN\nVector', 'EGNN\nVector+TDA'])
    ax.set_title(f'{title} ({unit})', fontsize=13)
    ax.set_ylabel('MAE (lower = better)')
    ax.grid(True, alpha=0.3, axis='y')
    for bar, val in zip(bars, v29_data[metric]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.01,
                f'{val:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.savefig('results/figures/v34/test_metrics_comparison.png', dpi=150, facecolor='white')
plt.show()
print('Saved: results/figures/v34/test_metrics_comparison.png')

## 3. Parameter Count Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6), constrained_layout=True)

models = ['FCNN', 'SchNet', 'EGNN', 'EGNN+TDA', 'EGNN\nVector', 'EGNN\nVector+TDA', 'EGNN\nTensor']
params = [53171, 500000, 951759, 952811, 968023, 981439, 941900]
colors = ['#607D8B', '#795548', '#2196F3', '#FF9800', '#4CAF50', '#9C27B0', '#F44336']

bars = ax.barh(models, params, color=colors, edgecolor='black')
ax.set_xlabel('Parameters', fontsize=13)
ax.set_title('Model Parameter Count Comparison', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

for bar, val in zip(bars, params):
    ax.text(bar.get_width() + 10000, bar.get_y() + bar.get_height()/2,
            f'{val:,}', ha='left', va='center', fontsize=11, fontweight='bold')

ax.set_xlim(0, max(params) * 1.15)
plt.savefig('results/figures/v34/parameter_count.png', dpi=150, facecolor='white')
plt.show()
print('Saved: results/figures/v34/parameter_count.png')

## 4. TDA Analysis — Why TDA Doesn't Help

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6), constrained_layout=True)
fig.suptitle('TDA Impact Analysis: Why TDA Degrades Performance', fontsize=14, fontweight='bold')

# Left: TDA feature zeros analysis
ax = axes[0]
categories = ['H₀ Betti\n(16 bins)', 'H₁ Betti\n(16 bins)', 'H₂ Betti\n(16 bins)',
              'Entropy\n(3 dims)', 'Mean pers\n(1 dim)']
total = [16, 16, 16, 3, 1]
always_zero = [0, 12, 16, 0, 0]  # приблизительно из анализа
nonzero = [t - z for t, z in zip(total, always_zero)]

x = range(len(categories))
ax.bar(x, nonzero, color='#4CAF50', label='Informative', edgecolor='black')
ax.bar(x, always_zero, bottom=nonzero, color='#F44336', label='Always zero', edgecolor='black')
ax.set_xticks(x); ax.set_xticklabels(categories, fontsize=10)
ax.set_ylabel('Number of features')
ax.set_title('TDA Feature Quality (52D total)', fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
ax.text(0.5, 0.95, '54% of features are ALWAYS ZERO', transform=ax.transAxes,
        ha='center', fontsize=12, fontweight='bold', color='#F44336',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# Right: EGNN vs EGNN+TDA metrics
ax = axes[1]
metrics = ['mu_mae', 'alpha_mae', 'gap_mae', 'test_loss']
metric_labels = ['μ MAE', 'α MAE', 'gap MAE', 'test_loss']
egnn_vals = [0.2446, 0.4597, 0.0046, 0.2854]
egnn_tda_vals = [0.2735, 0.4977, 0.0053, 0.3210]

x = np.arange(len(metrics))
width = 0.35
ax.bar(x - width/2, egnn_vals, width, label='EGNN', color='#2196F3', edgecolor='black')
ax.bar(x + width/2, egnn_tda_vals, width, label='EGNN+TDA', color='#FF9800', edgecolor='black')
ax.set_xticks(x); ax.set_xticklabels(metric_labels)
ax.set_ylabel('MAE / Loss (lower = better)')
ax.set_title('EGNN vs EGNN+TDA (v29, bs=512)', fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# Add delta arrows
for i, (e, t) in enumerate(zip(egnn_vals, egnn_tda_vals)):
    delta_pct = (t - e) / e * 100
    ax.annotate(f'+{delta_pct:.0f}%', xy=(i + width/2, t),
                xytext=(i + width/2, t * 1.1),
                ha='center', fontsize=10, color='#F44336', fontweight='bold')

plt.savefig('results/figures/v34/tda_analysis.png', dpi=150, facecolor='white')
plt.show()
print('Saved: results/figures/v34/tda_analysis.png')

## 5. EGNN vs EGNN+TDA — Training Comparison

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5), constrained_layout=True)
fig.suptitle('EGNN vs EGNN+TDA: Training Dynamics (v29, bs=512)', fontsize=14, fontweight='bold')

for ax, (col, title) in zip(axes, [
    ('val_loss', 'Val Loss'),
    ('val_mu_mae', 'μ MAE (Debye)'),
    ('val_alpha_mae', 'α MAE (Bohr³)'),
    ('val_gap_mae', 'gap MAE (Hartree)')
]):
    egnn_df = histories['egnn']
    tda_df = histories['egnn_tda']
    if col in egnn_df.columns:
        ax.plot(egnn_df['epoch'], egnn_df[col], color='#2196F3', label='EGNN', linewidth=2)
    if col in tda_df.columns:
        ax.plot(tda_df['epoch'], tda_df[col], color='#FF9800', label='EGNN+TDA', linewidth=2)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Epoch')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.savefig('results/figures/v34/egnn_vs_tda_training.png', dpi=150, facecolor='white')
plt.show()
print('Saved: results/figures/v34/egnn_vs_tda_training.png')

## 6. Model Architecture Comparison Table

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6), constrained_layout=True)
ax.axis('off')

table_data = [
    ['Model', 'Equivariance', 'μ output', 'α output', 'TDA', 'Params'],
    ['FCNN', 'None', 'scalar', 'scalar', 'No', '53K'],
    ['SchNet', 'Trans + Perm', 'scalar', 'scalar', 'No', '500K'],
    ['EGNN', 'E(3)', 'scalar', 'scalar', 'No', '952K'],
    ['EGNN+TDA', 'E(3)', 'scalar', 'scalar', 'Yes', '953K'],
    ['EGNN Vector', 'E(3)', 'vector R³', 'scalar', 'No', '968K'],
    ['EGNN Vector+TDA', 'E(3)', 'vector R³', 'scalar', 'Yes', '981K'],
    ['EGNN Tensor', 'E(3)', 'vector R³', 'tensor R^(3×3)', 'No', '942K'],
    ['EGNN Tensor+TDA', 'E(3)', 'vector R³', 'tensor R^(3×3)', 'Yes', '~960K'],
]

table = ax.table(cellText=table_data[1:], colLabels=table_data[0],
                 cellLoc='center', loc='center', colWidths=[0.18, 0.15, 0.12, 0.15, 0.08, 0.1])
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.2, 1.8)

# Color header
for j in range(6):
    table[0, j].set_facecolor('#2196F3')
    table[0, j].set_text_props(color='white', fontweight='bold')

# Color rows
colors_row = ['#E3F2FD', '#FFF3E0', '#E8F5E9', '#FFF3E0', '#F3E5F5', '#FFF3E0', '#FFEBEE', '#FFF3E0']
for i, color in enumerate(colors_row):
    for j in range(6):
        table[i + 1, j].set_facecolor(color)

ax.set_title('Model Architecture Comparison (8 models)', fontsize=14, fontweight='bold', pad=20)
plt.savefig('results/figures/v34/model_comparison_table.png', dpi=150, facecolor='white', bbox_inches='tight')
plt.show()
print('Saved: results/figures/v34/model_comparison_table.png')

## 7. Final Results Summary

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8), constrained_layout=True)
ax.axis('off')

table_data = [
    ['Model', 'Version', 'Epochs', 'μ MAE\n(Debye)', 'α MAE\n(Bohr³)', 'gap MAE\n(Hartree)', 'Test Loss'],
    ['FCNN', 'v26.1 bs=256', '176', '0.851', '2.271', '0.0261', '1.235'],
    ['SchNet', 'v26.1 bs=256', '226', '0.131', '0.442', '0.0033', '0.186'],
    ['EGNN', 'v26.1 bs=256', '131', '0.179', '0.393', '0.0041', '0.227'],
    ['EGNN', 'v29 bs=512', '164', '0.245', '0.460', '0.0046', '0.285'],
    ['EGNN+TDA', 'v29 bs=512', '179', '0.274', '0.498', '0.0053', '0.321'],
    ['EGNN Vector', 'v29 bs=512', '140', '0.710', '0.390', '0.0041', '0.575'],
    ['EGNN Vector+TDA', 'v29 bs=512', '145', '0.704', '0.357', '0.0039', '0.565'],
    ['EGNN', 'v33.10 bs=1024', '110', '0.286', '0.574', '0.0058', '0.344'],
    ['EGNN Tensor', 'v33.10 bs=1024', '13*', '3.880', '11.941', '0.0392', '2.303'],
]

table = ax.table(cellText=table_data[1:], colLabels=table_data[0],
                 cellLoc='center', loc='center')
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.0, 1.6)

for j in range(7):
    table[0, j].set_facecolor('#37474F')
    table[0, j].set_text_props(color='white', fontweight='bold')

# Highlight best results
best_rows = {3: '#E8F5E9', 4: '#FFF3E0'}  # SchNet and EGNN v26.1
for i in range(len(table_data) - 1):
    if i in best_rows:
        for j in range(7):
            table[i + 1, j].set_facecolor(best_rows[i])

ax.set_title('Final Results Summary (All Versions)', fontsize=14, fontweight='bold', pad=20)
ax.text(0.5, -0.05, '* EGNN Tensor stopped after 13 epochs (Kaggle 12h limit). Full training in progress.',
        transform=ax.transAxes, ha='center', fontsize=10, style='italic', color='gray')

plt.savefig('results/figures/v34/final_results_summary.png', dpi=150, facecolor='white', bbox_inches='tight')
plt.show()
print('Saved: results/figures/v34/final_results_summary.png')

## 8. Key Findings

In [ ]:
findings = """
=== KEY FINDINGS ===

1. BEST MODEL: SchNet (v26.1, bs=256) — μ MAE=0.131, α MAE=0.442
   - SchNet outperforms EGNN on μ and α
   - EGNN is better on gap (0.0041 vs 0.0033)

2. TDA DOES NOT HELP:
   - EGNN+TDA is WORSE than EGNN on ALL metrics (+12-26%)
   - 54% of TDA features (28/52) are ALWAYS ZERO
   - After BatchNorm, zero features become noise → degrades model
   - Alchemy molecules are small (9-29 atoms) — topology too simple
   - This is a VALID NEGATIVE RESULT

3. EGNN VECTOR needs more training:
   - μ MAE=0.710 (vs 0.245 for scalar EGNN) — 2.9x worse
   - Vector output requires learning partial charges → slower convergence
   - But α MAE=0.390 (BETTER than scalar EGNN's 0.460)

4. BATCH SIZE MATTERS:
   - bs=256 (v26.1): EGNN μ MAE=0.179
   - bs=512 (v29):    EGNN μ MAE=0.245 (+37%)
   - bs=1024 (v33):   EGNN μ MAE=0.286 (+60%)
   - Larger batch = fewer gradient steps per epoch = worse convergence

5. EGNN TENSOR (Part B):
   - Only 13 epochs completed (Kaggle 12h limit)
   - Predicts full vector μ and tensor α via partial charges
   - Needs 50-100+ epochs to converge (zero-init charges)
   - Training in progress on 100 epochs
"""
print(findings)